In [1]:
from rdkit import Chem
from glob import glob
import pandas as pd
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.error')  
pd.set_option("display.max_colwidth", None)

This Jupyter notebook combines all Selleckchem libraries into one, removes duplicates, and eliminates unreadable SMILES.

In [2]:
df_list = []
for xlsx_file in glob(r"C:\Users\USER\OneDrive\Documents\work\Projects\Selleckchem preparation\selleckchem_xlsx\*.xlsx"):
    df = pd.read_excel(xlsx_file, sheet_name=1)[["Name", "CAS Number", "URL", "SMILES"]]
    df_list.append(df)

df = pd.concat(df_list)
print(f"Number of SMILES entries: {df.shape[0]}")
print(f"Deleted nan SMILES: {df['SMILES'].isna().sum()}")
df = df[~df["SMILES"].isna()]
mask = df["URL"].str.contains("selleck").fillna(False)
df = pd.concat([df[mask], df[~mask]])
df = df[~df["SMILES"].duplicated()]
print(f"Number of unique SMILES: {df.shape[0]}")
df["Mol"] = df["SMILES"].apply(Chem.MolFromSmiles)
print(f"Deleted bad SMILES: {df['Mol'].isna().sum()}")
df = df[~df["Mol"].isna()]
print(f"Final number of SMILES to save: {df.shape[0]}")
df[["Name", "CAS Number", "SMILES", "URL"]].to_csv(r"C:\Users\USER\OneDrive\Documents\work\Projects\Selleckchem preparation\selleckchem_smiles.csv", index=False)

Number of SMILES entries: 91437
Deleted nan SMILES: 168
Number of unique SMILES: 10106
Deleted bad SMILES: 122
Final number of SMILES to save: 9984
